In [3]:
# triad_modulation_analysis.ipynb

import numpy as np
import pandas as pd
from pathlib import Path
import triad_utils as tu
from triad_utils import *

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    plt = None
    sns = None

# Load data
M = pd.read_csv('mij_matrix.csv', index_col=0)
labels = M.index.tolist()

# Infer E/I labels from connection weights
is_excitatory = (M > 0).any(axis=1)
is_inhibitory = (M < 0).any(axis=1)

# Variants
variants = {
    'with_self': M.values.copy(),
    'no_self': M.values.copy()
}
np.fill_diagonal(variants['no_self'], 0)

NUM_NULL_SHUFFLES = 100

motif_counts = tu.motif_counts(variants, is_excitatory, is_inhibitory)
HAS_PLOTS = plt is not None and all(
    name in globals()
    for name in [
        'plot_triad_enrichment',
        'plot_enrichment_vs_perturbation',
        'plot_regional_triad_distribution',
        'plot_schur_spectra_comparison',
        'plot_candidate_motifs',
    ]
)

# Analysis output directory
OUTPUT_DIR = Path('outputs/triad_modulation_analysis/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
# ============================================================================
# ENRICHMENT ANALYSIS
# ============================================================================

enrichment_stats = {}
for variant, matrix in variants.items():
    enrichment_stats[variant] = assess_triad_enrichment(
        matrix,
        is_excitatory,
        is_inhibitory,
        observed_motif_counts=motif_counts[variant],
        num_shuffles=NUM_NULL_SHUFFLES,
        random_seed=42,
    )
    print(f"Triad enrichment analysis ({variant}):")
    display(enrichment_stats[variant])

# Plot enrichment
if HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)
    plot_triad_enrichment(enrichment_stats, ax=ax)
    fig.savefig(OUTPUT_DIR / 'triad_enrichment.png', dpi=300, bbox_inches='tight')

NameError: name 'motif_counts' is not defined

In [ ]:
# ============================================================================
# PERTURBATION ANALYSIS
# ============================================================================

perturbation_results = {}
for variant, matrix in variants.items():
    perturbation_results[variant] = measure_triad_perturbations(
        matrix, motif_counts[variant], is_excitatory, is_inhibitory
    )
    print(f"Triad perturbation analysis ({variant}):")
    display(perturbation_results[variant])

In [ ]:
# ============================================================================
# REGIONAL ANALYSIS
# ============================================================================

region_map = assign_nodes_to_regions(labels)

regional_triad_counts = {}
for variant, matrix in variants.items():
    regional_triad_counts[variant] = count_triads_by_region(
        motif_counts[variant], region_map, matrix, is_excitatory
    )
    print(f"Regional triad counts ({variant}):")
    display(regional_triad_counts[variant])

In [ ]:
# ============================================================================
# SCHUR ANALYSIS
# ============================================================================

schur_stats = {}
for variant, matrix in variants.items():
    schur_stats[variant] = analyze_triad_schur_complements(
        matrix, motif_counts[variant], is_excitatory, is_inhibitory
    )
    print(f"Triad Schur analysis ({variant}):")
    display(schur_stats[variant])


In [ ]:
# ============================================================================
# CANDIDATE MOTIF NOMINATION
# ============================================================================

candidate_motifs = {}
for variant in variants:
    candidate_motifs[variant] = nominate_stabilizing_triads(
        enrichment_stats[variant], perturbation_results[variant], schur_stats[variant]
    )
    print(f"Candidate stabilizing triad motifs ({variant}):")
    display(candidate_motifs[variant])

In [ ]:

# ============================================================================
# SUMMARY PLOTS
# ============================================================================

if HAS_PLOTS:
    # (Code for summary plots integrating findings)
    # ...
    # Example:
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), dpi=100)
    plot_enrichment_vs_perturbation(enrichment_stats, perturbation_results, ax=axes[0,0])
    plot_regional_triad_distribution(regional_triad_counts, ax=axes[0,1])
    plot_schur_spectra_comparison(schur_stats, ax=axes[1,0])
    plot_candidate_motifs(candidate_motifs, ax=axes[1,1])
    fig.savefig(OUTPUT_DIR / 'triad_analysis_summary.png', dpi=300, bbox_inches='tight')